In [1]:
import pandas as pd

df = pd.read_csv("churn_data.csv")

df.head()

,age,salaire,abonnement,anciennete,consommation,churn
0,56,4543,Premium,12,77,0
1,69,6508,Premium,0,83,0
2,46,5270,Standard,9,21,0
3,32,5999,Premium,4,74,0
4,60,2670,Premium,11,26,0


In [2]:
df.shape

(500, 6)

In [3]:
df.describe()

,age,salaire,anciennete,consommation,churn
count,500.000000,500.00000,500.000000,500.000000,500.000000
mean,44.220000,4666.30800,7.144000,48.660000,0.038000
std,15.036082,1844.86557,4.364558,28.927115,0.191388
min,18.000000,1209.00000,0.000000,0.000000,0.000000
25%,32.000000,3124.75000,3.000000,23.750000,0.000000
50%,45.000000,4679.00000,7.000000,47.000000,0.000000
75%,57.000000,6225.25000,11.000000,74.250000,0.000000
max,69.000000,7988.00000,14.000000,99.000000,1.000000


In [4]:
df.isnull().sum()

age             0
salaire         0
abonnement      0
anciennete      0
consommation    0
churn           0
dtype: int64

In [5]:
df["churn"].value_counts()

churn
0    481
1     19
Name: count, dtype: int64

In [6]:
from sklearn.model_selection import train_test_split

# ============================================================
# 1. Conversion de la colonne "abonnement"
# ============================================================
# Les algorithmes de Machine Learning travaillent avec des nombres.
# On convertit donc les valeurs textuelles de la colonne "abonnement"
# en valeurs numériques.

df["abonnement"] = df["abonnement"].map({
    "Basique": 0,
    "Standard": 1,
    "Premium": 2
})

# ============================================================
# 2. Définition des variables explicatives (features)
# ============================================================
# X contient les variables utilisées pour faire les prédictions.

X = df[
    [
        "age",
        "salaire",
        "abonnement",
        "anciennete",
        "consommation"
    ]
]

# ============================================================
# 3. Définition de la variable cible (target)
# ============================================================
# y contient la variable que le modèle doit prédire.
# 0 = client qui reste
# 1 = client qui part

y = df["churn"]

# ============================================================
# 4. Séparation des données en Train / Test
# ============================================================
# 80% des données sont utilisées pour entraîner le modèle.
# 20% sont conservées pour tester le modèle.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,      # 20% des données pour le test
    random_state=42     # garantit la reproductibilité
)

# ============================================================
# 5. Vérification des dimensions
# ============================================================

print("Taille de X_train :", X_train.shape)
print("Taille de X_test  :", X_test.shape)
print("Taille de y_train :", y_train.shape)
print("Taille de y_test  :", y_test.shape)

Taille de X_train : (400, 5)
Taille de X_test  : (100, 5)
Taille de y_train : (400,)
Taille de y_test  : (100,)


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# ============================================================
# 1. Création du modèle Random Forest
# ============================================================
# On crée une forêt composée de 100 arbres de décision.
# Chaque arbre vote pour une classe et la classe majoritaire
# est retenue comme résultat final.
#
# max_depth=5 limite la profondeur des arbres afin de réduire
# le risque de surapprentissage (overfitting).

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

# ============================================================
# 2. Entraînement du modèle
# ============================================================
# .fit() lance l'apprentissage.
# Le modèle analyse X_train et y_train afin d'apprendre
# la relation entre les caractéristiques des clients et le churn.

model.fit(X_train, y_train)

print("Modèle Random Forest entraîné avec succès !")

# ============================================================
# 3. Prédiction sur les données de test
# ============================================================
# Le modèle prédit le churn des 100 clients de test.
# Ces données n'ont pas été utilisées pendant l'entraînement.

y_pred = model.predict(X_test)

# ============================================================
# 4. Évaluation du modèle
# ============================================================

# Accuracy : pourcentage de prédictions correctes.
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy :", accuracy)

# ============================================================
# 5. Rapport de classification
# ============================================================
# Le classification_report affiche notamment :
# - precision : précision des prédictions
# - recall    : capacité à retrouver les vrais cas
# - f1-score  : moyenne harmonique entre precision et recall
# - support   : nombre d'exemples dans chaque classe

print("\nRapport de classification :")
print(classification_report(y_test, y_pred))

Modèle Random Forest entraîné avec succès !
Accuracy : 0.99

Rapport de classification :
              precision    recall  f1-score   support

           0       0.99      1.00      0.99        95
           1       1.00      0.80      0.89         5

    accuracy                           0.99       100
   macro avg       0.99      0.90      0.94       100
weighted avg       0.99      0.99      0.99       100



In [8]:
import joblib

# ============================================================
# Sauvegarde du modèle
# ============================================================
# joblib.dump() permet de sauvegarder le modèle entraîné
# dans un fichier binaire.
#
# Le fichier "model.pkl" pourra ensuite être utilisé
# pour faire des prédictions sans réentraîner le modèle.

joblib.dump(model, "model.pkl")

print("Modèle sauvegardé avec succès dans : model.pkl")

Modèle sauvegardé avec succès dans : model.pkl


In [9]:
# ============================================================
# ESSAI 2 : un modèle plus simple
# ============================================================

model = RandomForestClassifier(
    n_estimators=50,   # moins d'arbres que dans l'Essai 1 : 100 -> 50
    max_depth=3,       # arbres moins profonds que dans l'Essai 1 : 5 -> 3
    random_state=42
)

# Entraînement du modèle
model.fit(X_train, y_train)

# Prédiction sur les données de test
y_pred = model.predict(X_test)

# Évaluation du modèle
print("Accuracy Essai 2 :", accuracy_score(y_test, y_pred))

# Sauvegarde du modèle
# Cette sauvegarde écrase le fichier model.pkl de l'Essai 1.
joblib.dump(model, "model.pkl")

print("model.pkl écrasé avec l'Essai 2")

Accuracy Essai 2 : 0.99
model.pkl écrasé avec l'Essai 2


In [10]:
# ============================================================
# ESSAI 3 : on retire la colonne "salaire"
# ============================================================

# On crée un nouveau jeu de features sans la colonne "salaire".
X = df[
    [
        "age",
        "abonnement",
        "anciennete",
        "consommation"
    ]
]

# Nouvelle séparation Train / Test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Création du modèle
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

# Entraînement
model.fit(X_train, y_train)

# Prédiction
y_pred = model.predict(X_test)

# Évaluation
print(
    "Accuracy Essai 3 (sans salaire) :",
    accuracy_score(y_test, y_pred)
)

# Sauvegarde du modèle
# Cette sauvegarde écrase le model.pkl de l'Essai 2.
joblib.dump(model, "model.pkl")

print("model.pkl écrasé avec l'Essai 3")


Accuracy Essai 3 (sans salaire) : 1.0
model.pkl écrasé avec l'Essai 3


In [11]:
# ============================================================
# Vérification : que contient réellement model.pkl ?
# ============================================================

# Chargement du dernier modèle sauvegardé
loaded = joblib.load("model.pkl")

# n_features_in_ indique le nombre de features attendues
# par le modèle.
#
# 5 -> modèle avec salaire (Essai 1 ou Essai 2)
# 4 -> modèle sans salaire (Essai 3)

print(
    "Nombre de features attendu :",
    loaded.n_features_in_
)

Nombre de features attendu : 4
